# Artifact flagging (Catalog 2)

A statistical outlier is not automatically an astrophysical one: a burst can rank as anomalous
because a fitted parameter is poorly constrained. But the opposite trap is just as real and was
made once on this project. sp_idx and sp_run were dropped as "unconstrained fits", then reinstated
once the CHIME Catalog 2 paper made clear that a large |sp_run| is produced BY DESIGN by
band-limited bursts and encodes narrowbandness, a defining repeater trait (Pleunis et al. 2021).
A large value with an error bar as large as itself can be a real extreme burst, not a failure.

So Phase 1 FLAGS, it does not remove. Each candidate from `04_candidate_validation_cat2.ipynb` is
annotated with the relative fit error of its driving feature (its most extreme value in the scaled
space the detector saw). A high relative error marks the candidate for inspection, not deletion.
The artifact-versus-real determination is deferred to Phase 2, where the dynamic spectrum
(waterfall) shows directly whether a burst is a genuine narrow or narrowband event or a fit glitch.
Nothing is discarded here, so no real anomaly can be lost to a heuristic.

Run `04_candidate_validation_cat2.ipynb` first so the shortlist on disk is current.

## Load the shortlist, the scaled features, and the raw catalogue

In [1]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT / 'src'))

import numpy as np
import pandas as pd

PROC = PROJECT / 'data' / 'processed' / 'phase_1'
shortlist = pd.read_csv(PROC / 'catalog2_candidate_shortlist.csv')
scaled = pd.read_csv(PROC / 'catalog2_features_scaled.csv')
raw = pd.read_csv(PROJECT / 'data' / 'raw' / 'chimefrbcat2.csv')

# The eight features the detector scored, and the six of them that carry a raw fit-error column.
FEATURE_COLS = ['dm_fitb', 'width_fitb', 'flux', 'fluence', 'sp_idx', 'sp_run', 'peak_freq', 'bandwidth']
ERR_COLS = {'dm_fitb': 'dm_fitb_err', 'width_fitb': 'width_fitb_err', 'flux': 'flux_err',
            'fluence': 'fluence_err', 'sp_idx': 'sp_idx_err', 'sp_run': 'sp_run_err'}

print(f'{len(shortlist)} candidate bursts to screen')
print(f'features with a raw error column: {list(ERR_COLS)}')
print(f'features without one (no per-burst reliability check): '
      f'{[c for c in FEATURE_COLS if c not in ERR_COLS]}')

36 candidate bursts to screen
features with a raw error column: ['dm_fitb', 'width_fitb', 'flux', 'fluence', 'sp_idx', 'sp_run']
features without one (no per-burst reliability check): ['peak_freq', 'bandwidth']


## Driving feature and relative fit error

For each candidate, the driving feature is its most extreme value in the scaled space the
detector actually used (largest absolute scaled value). The relative fit error of each
error-bearing feature is the raw error divided by the absolute raw value; a value at or above 1
means the error bar is as large as the measurement, i.e. an unconstrained fit.

In [2]:
key = ['tns_name', 'sub_num']
scaled_idx = scaled.set_index(key)
raw_idx = raw.set_index(key)

records = []
for _, row in shortlist.iterrows():
    k = (row['tns_name'], row['sub_num'])
    sc = scaled_idx.loc[k, FEATURE_COLS].astype(float)
    driver = sc.abs().idxmax()                     # most extreme feature in the scaled space

    rel_errs = {}
    for feat, ecol in ERR_COLS.items():
        val = float(raw_idx.loc[k, feat])
        err = float(raw_idx.loc[k, ecol])
        rel_errs[feat] = abs(err) / abs(val) if val != 0 else np.inf

    driver_rel = rel_errs.get(driver, np.nan)      # NaN if driver has no error column
    records.append({'tns_name': row['tns_name'], 'sub_num': row['sub_num'],
                    'n_methods': row['n_methods'], 'driver': driver,
                    'driver_rel_err': driver_rel,
                    'max_rel_err': max(rel_errs.values())})

screen = pd.DataFrame(records)
print('Candidate drivers and fit reliability (driver_rel_err = error/value of the driving feature):')
print(screen.round(2).to_string(index=False))

Candidate drivers and fit reliability (driver_rel_err = error/value of the driving feature):
    tns_name  sub_num  n_methods     driver  driver_rel_err  max_rel_err
FRB20201126C        0          3       flux            0.60         0.60
FRB20200130B        0          3 width_fitb            2.17         2.17
FRB20221219D        0          3     sp_run            0.11         0.63
FRB20220407A        0          3 width_fitb           38.41        38.41
FRB20211108A        1          3 width_fitb            5.74         5.74
FRB20210513C        0          3  peak_freq             NaN         2.33
FRB20210512C        0          3     sp_run            0.39         0.61
FRB20210423B        0          3 width_fitb            8.16         8.16
FRB20210222C        0          3     sp_run            0.35         0.93
FRB20200723B        0          3    fluence            0.26         0.29
FRB20200705D        0          3     sp_run            0.38         2.81
FRB20200429A        0          

## Flagging

Each candidate is annotated, not filtered. A `fit_flag` of `inspect_fit` means the driving feature
has a relative fit error at or above `INSPECT_THRESHOLD` (the error bar is as large as the value),
so the anomaly should be checked against the waterfall in Phase 2 before being trusted as real
morphology. `no_error_column` marks candidates driven by peak_freq or bandwidth, which carry no raw
error column. All candidates are retained.

In [3]:
# Phase 1 flags rather than removes. A large relative fit error is annotated for inspection, not
# used to drop a candidate: for a width near the time resolution, or a narrowband burst (large
# |sp_run| by design, a defining repeater trait, Pleunis et al. 2021), a large relative error is the
# expected signature of a REAL extreme burst, not a fit failure. Dropping on this heuristic is the
# 2026-06-17 mistake (sp_idx/sp_run were wrongly cut, then reinstated). The artifact-versus-real
# call is made in Phase 2 on the dynamic spectra. So all candidates are kept; poorly-constrained
# drivers are flagged for inspection.
INSPECT_THRESHOLD = 1.0


def fit_flag(r):
    if pd.isna(r['driver_rel_err']):
        return 'no_error_column'
    return 'inspect_fit' if r['driver_rel_err'] >= INSPECT_THRESHOLD else 'ok'


screen['fit_flag'] = screen.apply(fit_flag, axis=1)
annotated = screen.sort_values(['n_methods', 'driver_rel_err'], ascending=[False, False])

print('Fit-reliability flags (nothing dropped):')
print(screen['fit_flag'].value_counts().to_string())
print()
print(annotated.round(2).to_string(index=False))

out = PROC / 'catalog2_candidate_shortlist_flagged.csv'
annotated.to_csv(out, index=False)
print(f'\nSaved flagged shortlist (all {len(annotated)} candidates kept) to {out.relative_to(PROJECT)}')

Fit-reliability flags (nothing dropped):
fit_flag
ok                 25
inspect_fit         9
no_error_column     2

    tns_name  sub_num  n_methods     driver  driver_rel_err  max_rel_err        fit_flag
FRB20220407A        0          3 width_fitb           38.41        38.41     inspect_fit
FRB20210423B        0          3 width_fitb            8.16         8.16     inspect_fit
FRB20211108A        1          3 width_fitb            5.74         5.74     inspect_fit
FRB20200127A        0          3 width_fitb            3.66         3.66     inspect_fit
FRB20200130B        0          3 width_fitb            2.17         2.17     inspect_fit
FRB20201126C        0          3       flux            0.60         0.60              ok
FRB20230826A        1          3 width_fitb            0.57         0.57              ok
FRB20200429A        0          3     sp_run            0.47         0.61              ok
FRB20210512C        0          3     sp_run            0.39         0.61          

## Result

All candidate bursts are retained as the Phase 1 answer, each annotated with whether its driving
feature is well constrained. This is the candidate set of morphologically anomalous FRBs in the
Catalog 2 tabulated parameters, surviving cross-method agreement, with fit reliability flagged for
the next phase rather than used to discard. Caveats for the report:

- A high relative fit error is a flag for inspection, not evidence of an artifact; for narrow
  bursts (width near the time resolution) and narrowband bursts (large |sp_run| by design) it is
  the expected signature of a real extreme event. The real call is made in Phase 2 on the dynamic
  spectra.
- Sensitivity is established only for the geometries injected in `03`: global, local, subspace and
  contextual anomalies, and (via CBLOF) resolvable collective anomalies.
- One blind spot remains: a small embedded micro-cluster (a tight family of a few near-identical
  odd bursts) is below the cluster method's resolution and would not be recovered.

The next phase moves from the tabulated parameters to the dynamic spectra (waterfalls) of these
candidates, where fit-flagged bursts are adjudicated and the morphology is examined directly.